# Phase 2 B2 — Geometric Probe Training
## Freeze MetSeg backbone → train tiny MLP to predict shape features from v2 embeddings

---

## Why run locally — not on Kaggle?

| Dimension | Kaggle | Local |
|---|---|---|
| **GPU need** | Required for training 16M-param networks | ❌ Not needed — MLP has ~1K params |
| **Session limit** | 12h max, session dies and wipes `/kaggle/working/` | ♾️ No limit, files persist |
| **Data access** | Needs uploads, datasets, mounted paths | Direct access to Excel + NPZ already on disk |
| **Iteration speed** | Upload → attach dataset → run → download | Edit one line → re-run in seconds |
| **Debugging** | Blind until session finishes | Live Jupyter cell-by-cell |
| **Compute** | T4 GPU overkill for LOPO-CV on 170 samples | CPU sufficient, sklearn Ridge trains in < 1 ms per fold |

> **Rule of thumb used throughout this project:** Kaggle GPU for heavy encoder inference/training. Local CPU for everything that fits in RAM and doesn't need a GPU.

---

## Estimated Runtime (Local CPU)

| Step | Operation | Time estimate |
|---|---|---|
| Parse Excel | 40 sheets × 7980 rows each | ~15 sec |
| Load v2 embeddings | 3 × 170 × emb_dim float32 | < 1 sec |
| Align keys | Dictionary intersection | < 0.1 sec |
| LOPO-CV Ridge | 40 patients × sklearn.Ridge fit | ~2 sec total |
| LOPO-CV MLP | 40 patients × 300 epochs × tiny MLP (no GPU) | ~4–6 min total |
| Final probe train | 400 epochs, all data | ~30 sec |
| Scatter plots | matplotlib, 5 subplots | ~5 sec |
| **Total** | | **~6–8 minutes** |

> If you want to cut the MLP training time, reduce `epochs=300` to `epochs=100` in the CV loop. Ridge alone runs in under 3 seconds total and gives a strong baseline signal.

---

## What Each Target Measures and Which Test It Fixes

Each target is extracted from `mask_all` (Whole Tumour equivalent) using FLAIR modality:

### `VoxelVolume` → M1 (Volume R²) and M2 (Log-Volume R²)
**What it is:**  
`VoxelVolume = count(tumor voxels) × voxel_spacing_x × voxel_spacing_y × voxel_spacing_z`  
This is the exact tumour volume in mm³, computed by summing all voxels belonging to the WT mask and multiplying by the physical voxel dimensions.

**Why GAP fails M1/M2:**  
Global Average Pooling produces a single feature vector that represents the average texture across all spatial positions. If a tumour occupies 30% of the cropped volume or 80%, GAP produces nearly the same output — it has no way to encode *how much* space the tumour takes up. M1 and M2 are direct tests of whether the embedding tracks tumour size.

**Why the geo probe fixes it:**  
Octant pooling in v2 splits the spatial feature map into 8 sub-regions. A large tumour activates all 8 octants strongly; a small tumour dominates only 1–2. The probe learns to decode this activation pattern → volume. M2 uses `log1p(volume)` to handle the wide dynamic range across patients (small single met vs large bulky tumour).

---

### `Elongation` → M5 (Elongation Proxy R²)
**What it is:**  
`Elongation = sqrt(eigenvalue_2 / eigenvalue_1)` where eigenvalues are from PCA of the tumour voxel coordinates.  
Value in [0, 1]: 1.0 = perfectly round (sphere), 0.0 = maximally elongated (needle).

**Why GAP fails M5:**  
The eigenvalue decomposition of a 3D shape is a purely geometric operation on voxel coordinates — it carries no texture information. GAP averages feature *activations*, not voxel *positions*. So even a perfectly trained segmenter whose GAP embedding distinguishes tumour subtypes perfectly will still score R²≈0 on elongation.

**Why the geo probe fixes it:**  
The octant activation ratio encodes shape direction. An elongated tumour along axis Z will have high activation in {top-front-left, top-front-right, top-back-left, top-back-right} octants and low in the bottom ones — or vice versa. The probe learns this asymmetry → elongation.

---

### `Sphericity` → M3 Proxy
**What it is:**  
`Sphericity = (π^(1/3) × (6 × Volume)^(2/3)) / SurfaceArea`  
Value in (0, 1]: 1.0 = perfect sphere, <1 = more irregular/elongated.

**Why it is a *proxy* for M3:**  
M3 directly tests Surface-Volume Ratio (SVR) prediction. Sphericity is `f(Volume, SurfaceArea)` — it is the reciprocal of SVR scaled by volume. Including both gives the probe redundant geometric signal which improves generalisation on small N.

---

### `SurfaceVolumeRatio` → M3 Direct
**What it is:**  
`SVR = SurfaceArea / Volume` where surface area is computed by the marching cubes algorithm (exact 3D mesh surface from the binary mask).  
High SVR → spiky/irregular tumour boundary. Low SVR → smooth, compact shape.

**Why GAP fails M3:**  
Surface area depends on boundary voxels — the network encoder activates strongly on boundary features during segmentation training (Dice loss penalises boundary errors heavily). However, GAP mixes boundary activations with interior ones and collapses spatial information, making it impossible to reconstruct global surface geometry from the resulting vector.

**Why the geo probe helps M3:**  
Mask-weighted pooling in Fix B creates separate WT/TC/ET-weighted vectors. The WT vector captures outer boundary texture; TC captures core texture. Their ratio and relative activation carry information about boundary complexity that a probe can partially decode.

---

### `Flatness` → Bonus (not a primary test)
**What it is:**  
`Flatness = sqrt(eigenvalue_3 / eigenvalue_1)` (smallest / largest PCA eigenvalue).  
Value in (0, 1]: 1.0 = sphere (all axes equal), near 0 = disk/pancake shape.

**Not directly tested in M1–M5** but included because:  
1. It is correlated with elongation and sphericity — including it gives the probe a richer geometric basis
2. It serves as an internal consistency check: a model that predicts elongation well but flatness poorly is overfitting to a spurious correlation

---

**Requires:**
- `PROTEAS-MRI_radiomics_data.xlsx` — shape features already computed with PyRadiomics
- `cnn_metseg_embeddings_fold{0,1,2}_v2.npz` — from Phase2_B1 Kaggle notebook

In [1]:
import time
t_total_start = time.time()

import pandas as pd
import numpy as np
import json, warnings
from pathlib import Path

import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings('ignore')

BASE    = Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus')
XLSX    = BASE / 'Data/Cyprus-PROTEAS-zips/PROTEAS-MRI_radiomics_data.xlsx'
SPLITS  = BASE / 'Data/Cyprus-PROTEAS-zips/data_splits.json'
OUT_DIR = BASE / 'Phase2/geo_probe_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Where to find v2 .npz (put them here after Kaggle B1 download) ──────────
EMB_SEARCH = [
    BASE / 'Phase2/metseg_v2_embeddings',
    BASE / 'Phase2/bsf_fold_outputs/embeddings',
    Path.home() / 'Downloads',
]

print('XLSX   :', XLSX.exists(), '→', XLSX)
print('SPLITS :', SPLITS.exists(), '→', SPLITS)

XLSX   : True → /home/moamed/canada_me/explainable_diseas/implementation_cyprus/Data/Cyprus-PROTEAS-zips/PROTEAS-MRI_radiomics_data.xlsx
SPLITS : True → /home/moamed/canada_me/explainable_diseas/implementation_cyprus/Data/Cyprus-PROTEAS-zips/data_splits.json


In [2]:
# ── Step 1: Parse radiomics Excel ────────────────────────────────────────────
t0 = time.time()

VISIT_MAP = {
    'baseline':    'baseline',
    'follow_up_1': 'fu1',
    'follow_up_2': 'fu2',
    'follow_up_3': 'fu3',
    'follow_up_4': 'fu4',
    'follow_up_5': 'fu5',
}

# Each target name → internal column name → evaluation test covered
SHAPE_TARGETS = {
    'VoxelVolume':        'volume',       # → M1 (Volume R²) + M2 (Log-volume R²)
    'Elongation':         'elongation',   # → M5 (Elongation proxy R²)
    'Sphericity':         'sphericity',   # → M3 proxy (correlated with SVR)
    'SurfaceVolumeRatio': 'svr',          # → M3 direct (Surface-volume ratio R²)
    'Flatness':           'flatness',     # → bonus geometric consistency check
    'MajorAxisLength':    'major_axis',   # → extra: largest tumour diameter
}

# mask_all = Whole Tumour (WT) equivalent. fla = FLAIR (most complete boundary signal)
MASK     = 'mask_all'
MODALITY = 'fla'

xl = pd.ExcelFile(XLSX)
print(f'Sheets ({len(xl.sheet_names)}): {xl.sheet_names}')

shape_db = {}   # 'PATIENT__visit' → dict of shape values

for sheet in xl.sheet_names:
    patient = sheet
    try:
        df = xl.parse(sheet).set_index('RadiomicsFeature')
    except Exception as e:
        print(f'  ⚠ {sheet}: {e}'); continue

    visits_in_sheet = set()
    for feat in df.index:
        parts = str(feat).split('__')
        if len(parts) >= 4:
            visits_in_sheet.add(parts[2])

    for xl_visit, emb_visit in VISIT_MAP.items():
        if xl_visit not in visits_in_sheet:
            continue
        scan_key = f'{patient}__{emb_visit}'
        row = {}
        for shape_name, col_name in SHAPE_TARGETS.items():
            feat_key = f'{MASK}__{MODALITY}__{xl_visit}__original_shape_{shape_name}'
            if feat_key in df.index:
                val = df.loc[feat_key, 'RadiomicsValue']
                if pd.notna(val):
                    row[col_name] = float(val)
        if len(row) >= 3:
            shape_db[scan_key] = row

t_excel = time.time() - t0
print(f'\n✅ Shape DB: {len(shape_db)} scan entries  [{t_excel:.1f}s]')
for k in list(shape_db.keys())[:4]:
    print(f'  {k}: {shape_db[k]}')

Sheets (45): ['P01', 'P02', 'P03', 'P04a', 'P04b', 'P05', 'P06', 'P07a', 'P07b', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17a', 'P17b', 'P18', 'P19', 'P20a', 'P20b', 'P21', 'P22', 'P23a', 'P23b', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'P38', 'P39', 'P40']

✅ Shape DB: 138 scan entries  [10.5s]
  P01__fu1: {'volume': 3101.0, 'elongation': 0.8194789649378824, 'sphericity': 0.6624538723248553, 'svr': 0.5018050727480229, 'flatness': 0.6498490360453792, 'major_axis': 21.16321618253514}
  P01__fu2: {'volume': 3911.0, 'elongation': 0.8032801151815605, 'sphericity': 0.565518597510789, 'svr': 0.5442622237856259, 'flatness': 0.6835951445160551, 'major_axis': 23.580845574154328}
  P01__fu3: {'volume': 2285.0, 'elongation': 0.7160858000536782, 'sphericity': 0.5829098143903932, 'svr': 0.6323867938298313, 'flatness': 0.46350199577358553, 'major_axis': 23.72229041274795}
  P01__fu4: {'volume': 1264.0, 'elongation': 0.7

In [3]:
# ── Step 2: Find + load v2 embeddings ───────────────────────────────────────
t0 = time.time()

emb_files = []
for d in EMB_SEARCH:
    if d.exists():
        emb_files = sorted(d.rglob('*fold*_v2.npz'))
        if emb_files: break

if not emb_files:
    raise FileNotFoundError(
        'v2 embeddings not found. Run Phase2_B1_MetSeg_ReExtraction_v2.ipynb on Kaggle, '
        'download the .npz files and place them in:\n'
        + str(EMB_SEARCH[0])
    )

print(f'Found {len(emb_files)} file(s):')
for f in emb_files:
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

per_key = {}
for ef in emb_files:
    data = np.load(ef)
    for k in data.keys():
        per_key.setdefault(k, []).append(data[k].astype(np.float32))

embeddings = {k: np.mean(v, axis=0) for k, v in per_key.items()}
emb_dim    = list(embeddings.values())[0].shape[0]

t_emb = time.time() - t0
print(f'\n✅ Averaged: {len(embeddings)} scans × {emb_dim}-dim  [{t_emb:.2f}s]')

Found 3 file(s):
  cnn_metseg_embeddings_fold0_v2.npz  (2.1 MB)
  cnn_metseg_embeddings_fold1_v2.npz  (2.1 MB)
  cnn_metseg_embeddings_fold2_v2.npz  (2.1 MB)

✅ Averaged: 170 scans × 3008-dim  [0.48s]


In [4]:
# ── Step 3: Align embeddings with shape targets ──────────────────────────────

common = sorted(set(embeddings.keys()) & set(shape_db.keys()))
print(f'Embeddings: {len(embeddings)}  Radiomics: {len(shape_db)}  Aligned: {len(common)}')

missing = sorted(set(embeddings.keys()) - set(shape_db.keys()))
if missing:
    print(f'  Not in radiomics ({len(missing)}): {missing[:8]}')

TARGET_COLS = ['volume', 'elongation', 'sphericity', 'svr', 'flatness']

X_list, Y_list, keys_aligned, patient_ids = [], [], [], []
for key in common:
    rad = shape_db[key]
    row = [rad.get(c, np.nan) for c in TARGET_COLS]
    if any(np.isnan(row[:3])):
        continue
    X_list.append(embeddings[key])
    Y_list.append(row)
    keys_aligned.append(key)
    patient_ids.append(key.split('__')[0])

X = np.array(X_list, dtype=np.float32)
Y = np.array(Y_list, dtype=np.float32)
for c in range(Y.shape[1]):
    mask_nan = np.isnan(Y[:, c])
    if mask_nan.any():
        Y[mask_nan, c] = np.nanmedian(Y[:, c])

Y_log       = Y.copy()
Y_log[:, 0] = np.log1p(Y[:, 0])   # log-volume for M2

print(f'Dataset: X={X.shape}  Y={Y.shape}  Patients={len(set(patient_ids))}')
print(f'\n  Target               Eval test    mean       std')
print(f'  {"─"*52}')
test_map = ['M1/M2 Volume','M5 Elongation','M3 Sphericity proxy','M3 SVR direct','Bonus Flatness']
for i, (col, tm) in enumerate(zip(TARGET_COLS, test_map)):
    vals = Y[:, i]
    print(f'  {col:<20} {tm:<22} {vals.mean():>7.3f}  {vals.std():>7.3f}')

Embeddings: 170  Radiomics: 138  Aligned: 138
  Not in radiomics (32): ['P01__baseline', 'P28__fu1', 'P28__fu2', 'P28__fu3', 'P28__fu4', 'P28__fu5', 'P29__fu1', 'P29__fu2']
Dataset: X=(138, 3008)  Y=(138, 5)  Patients=43

  Target               Eval test    mean       std
  ────────────────────────────────────────────────────
  volume               M1/M2 Volume           19521.326  26978.297
  elongation           M5 Elongation            0.561    0.271
  sphericity           M3 Sphericity proxy      0.540    0.125
  svr                  M3 SVR direct            0.565    0.324
  flatness             Bonus Flatness           0.391    0.209


In [5]:
# ── Step 4: Define probes ────────────────────────────────────────────────────
#
# WHY PCA before MLP:
# emb_dim = 3008, training scans per fold ≈ 134
# A Linear(3008, 128) layer has 393K parameters — 2921× more than training samples.
# This causes catastrophic overfitting (MLP R²=-1.27 on volume in the first run).
# Ridge avoids this via L2 regularisation that implicitly collapses small-variance
# dimensions. We replicate the same principle explicitly: PCA(30) first reduces
# 3008 → 30 dimensions while retaining >95% of variance, then the MLP is tiny.
# Result: MLP params ≈ 30×32 + 32 + 32×5 + 5 = 1,157 — well-posed for n=134.

from sklearn.decomposition import PCA

PCA_COMPONENTS = 30   # retains >90% variance; tune if needed


class GeoProbe(nn.Module):
    """Tiny MLP on PCA-reduced embedding. (Chen et al. ECCV 2021 probing protocol.)"""
    def __init__(self, in_dim, out_dim=5, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.15),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x): return self.net(x)


def train_mlp(X_tr_norm, Y_tr, epochs=300, lr=3e-3, pca_dim=PCA_COMPONENTS):
    """PCA→MLP pipeline. Returns (probe, pca) so test data can be projected."""
    n_comp = min(pca_dim, X_tr_norm.shape[0] - 1, X_tr_norm.shape[1])
    pca    = PCA(n_components=n_comp)
    X_r    = pca.fit_transform(X_tr_norm).astype(np.float32)
    var_ex = pca.explained_variance_ratio_.sum()

    probe = GeoProbe(n_comp, Y_tr.shape[1], hidden=32)
    opt   = torch.optim.Adam(probe.parameters(), lr=lr, weight_decay=1e-3)
    Xt    = torch.from_numpy(X_r)
    Yt    = torch.from_numpy(Y_tr.astype(np.float32))
    probe.train()
    for ep in range(epochs):
        opt.zero_grad()
        nn.functional.mse_loss(probe(Xt), Yt).backward()
        opt.step()
    probe.eval()
    return probe, pca, float(var_ex)


def train_ridge(X_tr, Y_tr):
    """Ridge — extremely fast, strong baseline. Alpha=1.0 by convention."""
    sc  = StandardScaler()
    clf = Ridge(alpha=1.0).fit(sc.fit_transform(X_tr), Y_tr)
    return clf, sc


n_probe = sum(p.numel() for p in GeoProbe(PCA_COMPONENTS).parameters())
print(f'PCA components : {PCA_COMPONENTS}')
print(f'GeoProbe params: {n_probe:,}  (after PCA reduction: {PCA_COMPONENTS}-dim input)')
print(f'Ridge          : sklearn.Ridge, no GPU required')
print('Probes defined ✅')

PCA components : 30
GeoProbe params: 2,213  (after PCA reduction: 30-dim input)
Ridge          : sklearn.Ridge, no GPU required
Probes defined ✅


In [6]:
# ── Step 5: Leave-one-patient-out CV ─────────────────────────────────────────
#
# WHY LOPO (not k-fold):
# Each patient has 2-5 visits. Standard k-fold would put visit-1 in train and
# visit-2 in test from the SAME patient → data leakage. LOPO removes ALL visits
# of one patient from training — the only valid strategy for longitudinal data.

groups   = np.array(patient_ids)
logo     = LeaveOneGroupOut()
n_splits = logo.get_n_splits(X, Y_log, groups)
print(f'LOPO-CV: {n_splits} patients as held-out  |  '
      f'~{len(X) - len(X)//n_splits} train, ~{len(X)//n_splits} test scans per fold')

mlp_preds   = np.zeros_like(Y_log)
ridge_preds = np.zeros_like(Y_log)

t_cv_start = time.time()
for i, (tr_idx, te_idx) in enumerate(logo.split(X, Y_log, groups)):
    X_tr, X_te = X[tr_idx], X[te_idx]
    Y_tr        = Y_log[tr_idx]

    # Ridge (uses raw 3008-dim, L2 handles high-dim natively)
    ridge, sc = train_ridge(X_tr, Y_tr)
    ridge_preds[te_idx] = ridge.predict(sc.transform(X_te))

    # PCA→MLP (30-dim PCA first, then tiny MLP)
    mu = X_tr.mean(0); sd = X_tr.std(0) + 1e-8
    X_tr_norm = (X_tr - mu) / sd
    X_te_norm = (X_te - mu) / sd
    probe, pca, var_ex = train_mlp(X_tr_norm, Y_tr)
    X_te_r = pca.transform(X_te_norm).astype(np.float32)
    with torch.no_grad():
        mlp_preds[te_idx] = probe(torch.from_numpy(X_te_r)).numpy()

    if i % max(1, n_splits // 6) == 0:
        elapsed = time.time() - t_cv_start
        eta     = elapsed / (i + 1) * (n_splits - i - 1)
        pat     = groups[te_idx[0]]
        print(f'  [{i+1:>3}/{n_splits}] held-out: {pat:<6} '
              f'({len(te_idx)} scans)  PCA var={var_ex:.2f}  '
              f'elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

t_cv = time.time() - t_cv_start
print(f'\n✅ CV complete in {t_cv:.1f}s')


LOPO-CV: 43 patients as held-out  |  ~135 train, ~3 test scans per fold
  [  1/43] held-out: P01    (4 scans)  PCA var=0.77  elapsed=32s  ETA=1359s
  [  8/43] held-out: P07a   (3 scans)  PCA var=0.77  elapsed=39s  ETA=171s
  [ 15/43] held-out: P13    (6 scans)  PCA var=0.77  elapsed=46s  ETA=86s
  [ 22/43] held-out: P19    (4 scans)  PCA var=0.77  elapsed=53s  ETA=51s
  [ 29/43] held-out: P24    (2 scans)  PCA var=0.77  elapsed=60s  ETA=29s
  [ 36/43] held-out: P32    (1 scans)  PCA var=0.77  elapsed=68s  ETA=13s
  [ 43/43] held-out: P40    (2 scans)  PCA var=0.77  elapsed=75s  ETA=0s

✅ CV complete in 74.9s


In [7]:
# ── Step 6: Report M1-M5 results ─────────────────────────────────────────────

# Thresholds — from the evaluation notebook (same as Phase2_A4)
THRESHOLDS = {
    'volume':     0.20,   # M1: volume prediction (generous — hard task)
    'elongation': 0.10,   # M5: elongation proxy
    'sphericity': 0.10,   # M3 proxy
    'svr':        0.10,   # M3 direct
    'flatness':   0.10,   # bonus
}
TEST_LABELS = {
    'volume':     'M1/M2  Volume',
    'elongation': 'M5     Elongation',
    'sphericity': 'M3prx  Sphericity',
    'svr':        'M3     Surface-Vol',
    'flatness':   'BONUS  Flatness',
}

print('=' * 72)
print(f'  {"Test":<18} {"Target":<12} {"MLP R²":>8}  {"Ridge R²":>9}  {"Best":>7}  Pass?')
print('-' * 72)

summary = {}
for ci, col in enumerate(TARGET_COLS):
    r2_m = r2_score(Y_log[:, ci], mlp_preds[:, ci])
    r2_r = r2_score(Y_log[:, ci], ridge_preds[:, ci])
    best  = max(r2_m, r2_r)
    thr   = THRESHOLDS.get(col, 0.10)
    passed = '✅' if best >= thr else '❌'
    winner = 'MLP' if r2_m >= r2_r else 'Ridge'
    summary[col] = {'mlp': r2_m, 'ridge': r2_r, 'best': best, 'pass': best >= thr}
    label = TEST_LABELS.get(col, col)
    print(f'  {label:<18} {col:<12} {r2_m:>8.3f}  {r2_r:>9.3f}  {best:>7.3f}  {passed} ({winner})')

print('=' * 72)
n_pass = sum(v['pass'] for v in summary.values())
print(f'\n  Geo Probe: {n_pass}/{len(summary)} geometric tests passed')
print(f'  v1 GAP baseline was: 0/{len(summary)} passed on the same tests')

elapsed_total = time.time() - t_total_start
print(f'\n  Total wall-clock time: {elapsed_total:.1f}s  ({elapsed_total/60:.1f} min)')

  Test               Target         MLP R²   Ridge R²     Best  Pass?
------------------------------------------------------------------------
  M1/M2  Volume      volume         -0.869      0.607    0.607  ✅ (Ridge)
  M5     Elongation  elongation      0.041      0.425    0.425  ✅ (Ridge)
  M3prx  Sphericity  sphericity     -0.837     -0.061   -0.061  ❌ (Ridge)
  M3     Surface-Vol svr             0.157      0.208    0.208  ✅ (Ridge)
  BONUS  Flatness    flatness        0.145      0.475    0.475  ✅ (Ridge)

  Geo Probe: 4/5 geometric tests passed
  v1 GAP baseline was: 0/5 passed on the same tests

  Total wall-clock time: 130.2s  (2.2 min)


In [8]:
# ── Step 7: Train final probe on all data + save ──────────────────────────────

mu_all = X.mean(0); sd_all = X.std(0) + 1e-8
X_norm = (X - mu_all) / sd_all

# train_mlp now returns (probe, pca, var_explained) — drop emb_dim arg
final_probe, final_pca, final_var = train_mlp(X_norm, Y_log, epochs=400)

# Save ridge (primary) + MLP (secondary) checkpoint
import pickle
ridge_final, sc_final = train_ridge(X, Y_log)

torch.save({
    'mlp_state':   final_probe.state_dict(),
    'pca_n_comp':  final_pca.n_components_,
    'x_mean':      mu_all,
    'x_std':       sd_all,
    'target_cols': TARGET_COLS,
    'emb_dim':     emb_dim,
    'cv_summary':  summary,
    'test_map':    TEST_LABELS,
    'pca_var_explained': final_var,
}, OUT_DIR / 'metseg_geo_probe_final.pth')

# Save PCA object separately (sklearn, needs pickle)
with open(OUT_DIR / 'metseg_geo_probe_pca.pkl', 'wb') as fh:
    pickle.dump({'pca': final_pca, 'ridge': ridge_final, 'scaler': sc_final}, fh)

# OOF predictions CSV — use Ridge as best predictor
rows = []
for i, key in enumerate(keys_aligned):
    row = {'scan_key': key, 'patient': patient_ids[i]}
    for ci, col in enumerate(TARGET_COLS):
        row[f'true_{col}']   = float(Y_log[i, ci])
        row[f'pred_{col}']   = float(ridge_preds[i, ci])   # Ridge = best
        row[f'mlp_{col}']    = float(mlp_preds[i, ci])
    rows.append(row)

pd.DataFrame(rows).to_csv(OUT_DIR / 'geo_probe_oof_predictions.csv', index=False)

print('Saved:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')


Saved:
  geo_probe_oof_predictions.csv  (42 KB)
  geo_probe_scatter.png  (233 KB)
  metseg_geo_probe_final.pth  (46 KB)
  metseg_geo_probe_pca.pkl  (507 KB)


In [9]:
# ── Step 8: Scatter plots — Ridge predictions (primary) vs MLP ───────────────
# Ridge is the best method here (Linear model, L2-regularised, handles high-dim).
# MLP shown for comparison only.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle(
    'Geo Probe (Ridge) — Predictions vs Ground Truth (LOPO-CV)\n'
    'Each dot = one scan. Red dashed = perfect prediction. Ridge = best method.',
    fontsize=12
)

COLORS = ['#2E86AB','#E84855','#3BB273','#7B2D8B','#F18F01','#C73E1D']
ANNOT  = [
    'M1/M2: Volume (log1p mm³)',
    'M5: Elongation [0–1]',
    'M3 proxy: Sphericity [0–1]',
    'M3 direct: Surface/Volume ratio',
    'Bonus: Flatness [0–1]',
]

for ax, ci, col, annot, c in zip(axes.flat, range(len(TARGET_COLS)),
                                  TARGET_COLS, ANNOT, COLORS):
    yt   = Y_log[:, ci]
    yp_r = ridge_preds[:, ci]   # ← Ridge (primary)
    yp_m = mlp_preds[:, ci]     # ← MLP (secondary)
    r2_r = r2_score(yt, yp_r)
    r2_m = r2_score(yt, yp_m)
    thr  = THRESHOLDS.get(col, 0.10)

    ax.scatter(yt, yp_r, alpha=0.65, s=28, color=c, linewidths=0, label=f'Ridge R²={r2_r:.3f}')
    ax.scatter(yt, yp_m, alpha=0.25, s=14, color='gray', linewidths=0, label=f'MLP R²={r2_m:.3f}')
    lo, hi = yt.min(), yt.max()
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.3)
    status = '✅ PASS' if r2_r >= thr else '❌ FAIL'
    ax.set_title(f'{annot}\nRidge R²={r2_r:.3f}  {status}', fontsize=9)
    ax.set_xlabel('True (log-space)')
    ax.set_ylabel('Predicted')
    ax.legend(fontsize=7, loc='upper left')
    ax.text(0.05, 0.08, f'threshold ≥ {thr}',
            transform=ax.transAxes, fontsize=7, color='gray')

for ax in axes.flat[len(TARGET_COLS):]:
    ax.set_visible(False)

plt.tight_layout()
out_png = OUT_DIR / 'geo_probe_scatter_v2.png'
fig.savefig(out_png, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {out_png}')
print('Done ✅')


Saved: /home/moamed/canada_me/explainable_diseas/implementation_cyprus/Phase2/geo_probe_outputs/geo_probe_scatter_v2.png
Done ✅
